In [ ]:
import os
from dotenv import load_dotenv

# from typing import Annotated, List, Optional, Dict, Literal
# from typing_extensions import TypedDict
from typing import Literal
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.agents import create_agent
# from langchain_core.tools import tool
# from langchain_core.language_models.chat_models import BaseChatModel
# from langchain_core.messages import HumanMessage, trim_messages

# from langgraph.types import Command
# from langgraph.graph import StateGraph, MessagesState, START, END

from tools import check_text_spelling, extract_citations, extract_connectives

In [ ]:
load_dotenv()
PROVIDER = os.getenv("PROVIDER")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MODELS = {
	"GROQ": {
		"LLM":"openai/gpt-oss-120b",
		"SLM":"qwen/qwen3.8-27b",
	},
	"GEMINI": {
		"LLM": "gemini-2.5-pro",
		"SLM": "gemma-4-26b-a4b-it"
	}
}

if (PROVIDER.upper() in ["GOOGLE", "GEMINI"]):
	orchestrator_model = ChatGoogleGenerativeAI(model=MODELS["GEMINI"]["LLM"], temperature=0.5, api_key=GEMINI_API_KEY, max_tokens=2048)
	specialist_model = ChatGoogleGenerativeAI(model=MODELS["GEMINI"]["SLM"], temperature=0.2, api_key=GEMINI_API_KEY, max_tokens=2048)
	revisor_model = ChatGoogleGenerativeAI(model=MODELS["GEMINI"]["LLM"], temperature=0.2, api_key=GEMINI_API_KEY, max_tokens=2048)
else:
	orchestrator_model = ChatGroq(model=MODELS["GROQ"]["LLM"], temperature=0.5, api_key=GROQ_API_KEY, max_tokens=2048)
	specialist_model = ChatGroq(model=MODELS["GROQ"]["SLM"], temperature=0.2, api_key=GROQ_API_KEY, max_tokens=2048)
	revisor_model = ChatGroq(model=MODELS["GROQ"]["LLM"], temperature=0.2, api_key=GROQ_API_KEY, max_tokens=2048)

## Supervisor Agent

### Especialist Agents

In [ ]:
class Justification(BaseModel):
	excerpt: str = Field(description="Trecho com o erro identificado passível de dedução de nota")
	reason: str = Field(description="Explicação da dedução relacionada à rúbrica da Competência corrigida")

class CompetenceResponseFormat(BaseModel):
	"""
	Saída estruturada contendo a nota (0 a 200 intervalada por 40)
	e a justificativa de dedução das notas com o trecho respectivo
	"""
	grade: Literal[0, 40, 80, 120, 160, 200] = Field(description="Nota discretizada em intervalos de 40")
	justification: list[Justification] = Field(description="Justificativas de dedução de nota, com trecho e explicação")

#### 1st Competence Team

Competência I - Norma Culta

Avalia o domínio da modalidade escrita formal da língua portuguesa.<br/>
Analisa conhecimento das convenções de escrita, precisão gramatical, construção sintática e escolha vocabular.

Aspectos avaliados
- **convenções da escrita:** acentuação, ortografia, uso de hífen, emprego de letras maiúsculas e minúsculas e separação silábica (translineação)
- **gramaticais:** regência verbal e nominal, concordância verbal e nominal, tempos e modos verbais, pontuação, paralelismos sintático, morfológico e semântico, emprego de pronomes e crase
- **escolha de registro:** adequação à modalidade escrita formal, isto é, ausência de uso de registro informal e/ou de marcas de oralidade
- **escolha vocabular:** emprego de vocabulário preciso, o que significa que as palavras selecionadas são usadas em seu sentido correto e são apropriadas ao contexto em que aparecem

---

| Pontuação	| Justificativa	|
|-----------|---|
|200 pontos | Demonstra excelente domínio da modalidade escrita formal da língua portuguesa e de escolha de registro. Desvios gramaticais ou de convenções da escrita serão aceitos somente como excepcionalidade e quando não caracterizarem reincidência. |
|160 pontos	| Demonstra bom domínio da modalidade escrita formal da língua portuguesa e de escolha de registro, com poucos desvios gramaticais e de convenções da escrita. |
|120 pontos	| Demonstra domínio mediano da modalidade escrita formal da língua portuguesa e de escolha de registro, com alguns desvios gramaticais e de convenções da escrita. |
| 80 pontos	| Demonstra domínio insuficiente da modalidade escrita formal da língua portuguesa, com muitos desvios gramaticais, de escolha de registro e de convenções da escrita. |
| 40 pontos	| Demonstra domínio precário da modalidade escrita formal da língua portuguesa, de forma sistemática, com diversificados e frequentes desvios gramaticais, de escolha de registro e de convenções da escrita. |
| 0 ponto	| Demonstra desconhecimento da modalidade escrita formal da língua portuguesa. |

In [ ]:
class C1ResponseFormat(CompetenceResponseFormat):
	competence_id: Literal[1]

C1_SYS_PROMPT = "Avalie este trecho de acordo com a competência 1 do ENEM"

C1_TOOLS = [check_text_spelling]

C1_AGENT = create_agent(
	model=specialist_model,
	tools=C1_TOOLS,
	system_prompt=C1_SYS_PROMPT,
	response_format=CompetenceResponseFormat
)

In [ ]:
res = C1_AGENT.invoke({
	"messages": [("user", "Minha terra tem palmeiras onde canta o ssbiá")]
})
res.keys()

In [ ]:
res['messages']

In [ ]:
C1_OUTPUT = C1ResponseFormat(competence_id=1, **res['structured_response'].model_dump())
print(C1_OUTPUT)

#### 2nd Competence Team

Competência II - Compreensão da Proposta e Repertório

Avalia a compreensão da proposta de redação e a aplicação de conceitos das várias áreas de conhecimento para desenvolver o tema, dentro dos limites estruturais do texto dissertativo-argumentativo. <br/>
Analisa a abordagem do tema, a adequação ao tipo textual e o uso de repertório sociocultural.

Aspectos avaliados
- **compreensão do tema:** abordagem completa do tema proposto, isto é, tanto do assunto geral quanto do recorte específico, sem tangenciá-lo nem fugir dele
- **tipo textual:** atendimento à estrutura dissertativo-argumentativa, com proposição (tese), argumentação (desenvolvimento) e conclusão, sem traços de outros tipos textuais (narração, descrição, etc.)
- **repertório sociocultural:** uso de informações, fatos, citações ou experiências vividas que contribuam como argumento para a discussão proposta. Para ser bem aproveitado, o repertório deve ser:
  1. **legitimado:** ancorado em área do conhecimento reconhecida e corretamente atribuído, e não baseado apenas nos textos motivadores
  2. **pertinente:** relacionado ao tema discutido
  3. **produtivo:** usado como argumento, com função argumentativa identificável e interpretado pelo participante, e não como "repertório de bolso", isto é, referência pronta, genérica e inserida apenas para demonstrar erudição
- **autoria:** ausência de cópia de trechos dos textos motivadores

---

| Pontuação | Justificativa |
|-----------|---|
| 200 pontos | Desenvolve o tema por meio de argumentação consistente, a partir de um repertório sociocultural produtivo, e apresenta excelente domínio do texto dissertativo-argumentativo. |
| 160 pontos | Desenvolve o tema por meio de argumentação consistente e apresenta bom domínio do texto dissertativo-argumentativo, com proposição, argumentação e conclusão. |
| 120 pontos | Desenvolve o tema por meio de argumentação previsível e apresenta domínio mediano do texto dissertativo-argumentativo, com proposição, argumentação e conclusão. |
| 80 pontos | Desenvolve o tema recorrendo à cópia de trechos dos textos motivadores ou apresenta domínio insuficiente do texto dissertativo-argumentativo, não atendendo à estrutura com proposição, argumentação e conclusão. |
| 40 pontos | Apresenta o assunto, tangenciando o tema, ou demonstra domínio precário do texto dissertativo-argumentativo, com traços constantes de outros tipos textuais. |
| 0 ponto | Fuga ao tema/não atendimento à estrutura dissertativo-argumentativa. Nestes casos, a redação recebe nota zero e é anulada. |

In [ ]:
class C2ResponseFormat(CompetenceResponseFormat):
	competence_id: Literal[2]

C2_SYS_PROMPT = "Avalie este trecho de acordo com a competência 2 do ENEM. O tema é 'Desafios para valorização do SUS'. Dê resposta curtíssima, máximo 10 palavras."

C2_TOOLS = [extract_citations]

C2_AGENT = create_agent(
	model=specialist_model,
	tools=C2_TOOLS,
	system_prompt=C2_SYS_PROMPT,
	response_format=CompetenceResponseFormat
)

In [ ]:
res = C2_AGENT.invoke({
	"messages": [("user", "Drauzio Varella afirma: 'ser ou não ser, eis a questão' para justificar o sucateamento do SUS")]
})
res.keys()

In [ ]:
res['messages']

In [ ]:
C2_OUTPUT = C2ResponseFormat(competence_id=2, **res['structured_response'].model_dump())
print(C2_OUTPUT)

#### 3rd Competence Team

Competência III - Seleção e Organização de Argumentos

Avalia como o participante seleciona, relaciona, organiza e interpreta informações, fatos, opiniões e argumentos em defesa de um ponto de vista. <br/>
Analisa a inteligibilidade do texto, isto é, sua coerência e a plausibilidade entre as ideias apresentadas, a partir do projeto de texto.

Aspectos avaliados
- **seleção de argumentos:** escolha de informações, fatos e opiniões pertinentes à defesa do ponto de vista, sem ideias irrelevantes, repetidas ou excessivas
- **relação de sentido entre as partes:** coerência entre introdução, desenvolvimento e conclusão, de modo que os argumentos convirjam para a defesa do ponto de vista
- **progressão temática:** encadeamento das ideias, em que cada parágrafo traz informações coerentes com o que foi discorrido anteriormente, sem repetições desnecessárias nem saltos temáticos
- **desenvolvimento dos argumentos:** explicitação, explicação ou exemplificação das informações (por meio de exemplos, dados, comparações, analogias, etc.), com a relevância das ideias e do ponto de vista explicitada
- **projeto de texto:** planejamento prévio que define quais argumentos serão mobilizados e em que ordem, o que se evidencia quando a introdução já delimita os argumentos que serão desenvolvidos
- **autoria:** seleção e interpretação próprias das informações, para além de reproduzir os argumentos dos textos motivadores

---

| Pontuação | Justificativa |
|-----------|---|
| 200 pontos | Apresenta informações, fatos e opiniões relacionados ao tema proposto, de forma consistente e organizada, configurando autoria, em defesa de um ponto de vista. |
| 160 pontos | Apresenta informações, fatos e opiniões relacionados ao tema, de forma organizada, com indícios de autoria, em defesa de um ponto de vista. |
| 120 pontos | Apresenta informações, fatos e opiniões relacionados ao tema, limitados aos argumentos dos textos motivadores e pouco organizados, em defesa de um ponto de vista. |
| 80 pontos | Apresenta informações, fatos e opiniões relacionados ao tema, mas desorganizados ou contraditórios e limitados aos argumentos dos textos motivadores, em defesa de um ponto de vista. |
| 40 pontos | Apresenta informações, fatos e opiniões pouco relacionados ao tema ou incoerentes e sem defesa de um ponto de vista. |
| 0 ponto | Apresenta informações, fatos e opiniões não relacionados ao tema e sem defesa de um ponto de vista. |

In [ ]:
class C3ResponseFormat(CompetenceResponseFormat):
	competence_id: Literal[3]

C3_SYS_PROMPT = "Avalie este trecho de acordo com a competência 3 do ENEM. O tema é 'Desafios para valorização do SUS'. Dê resposta curtíssima, máximo 10 palavras."

C3_TOOLS = []

C3_AGENT = create_agent(
	model=specialist_model,
	# tools=C3_TOOLS,
	system_prompt=C3_SYS_PROMPT,
	response_format=CompetenceResponseFormat
)

In [ ]:
res = C3_AGENT.invoke({
	"messages": [("user", "Drauzio Varella afirma: 'ser ou não ser, eis a questão' para justificar o sucateamento do SUS")]
})
res.keys()

In [ ]:
res['messages']

In [ ]:
C3_OUTPUT = C3ResponseFormat(competence_id=3, **res['structured_response'].model_dump())
print(C3_OUTPUT)

#### 4th Competence Team

Competência IV - Mecanismos Linguísticos

Avalia o conhecimento dos mecanismos linguísticos necessários para a construção da argumentação, isto é, a estruturação lógica e formal entre as partes do texto. <br/>
Analisa a coesão textual, ou seja, os recursos linguísticos que promovem o encadeamento das ideias entre orações, períodos e parágrafos.

Aspectos avaliados
- **estruturação dos parágrafos:** desenvolvimento de cada parágrafo a partir de uma ideia principal à qual se ligam ideias secundárias (por comparação, por causa e consequência, por exemplificação, por detalhamento, etc.), com articulação explícita entre um parágrafo e outro
- **estruturação dos períodos:** construção de períodos complexos, formados por duas ou mais orações, capazes de expressar relações de causa e consequência, contradição, temporalidade, comparação, conclusão, etc.
- **referenciação:** retomada de pessoas, coisas, lugares e fatos ao longo do texto por meio de pronomes, advérbios, artigos, sinônimos, antônimos, hipônimos, hiperônimos, expressões resumitivas, metafóricas ou metadiscursivas, e também de elipse
- **operadores argumentativos e conectores:** uso de preposições, conjunções, pronomes relativos, advérbios e locuções adverbiais que estabelecem relações lógicas e semânticas (igualdade, adversidade, causa, consequência, conclusão, etc.) entre orações, períodos e parágrafos
- **diversidade e adequação dos recursos coesivos:** variedade de recursos empregados e relação lógica adequada entre as ideias que eles conectam. A coesão não depende da mera presença de conectivos, e o uso excessivo ou formulaico deles não é critério de qualidade
- **inadequações a evitar:** ausência de articulação entre orações, frases e parágrafos, ausência de paragrafação, emprego de conector sem relação lógica com o contexto, e repetição ou substituição inadequada de palavras sem recorrer a recursos anafóricos

---

| Pontuação | Justificativa |
|-----------|---|
| 200 pontos | Articula bem as partes do texto e apresenta repertório diversificado de recursos coesivos. |
| 160 pontos | Articula as partes do texto, com poucas inadequações, e apresenta repertório diversificado de recursos coesivos. |
| 120 pontos | Articula as partes do texto, de forma mediana, com inadequações, e apresenta repertório pouco diversificado de recursos coesivos. |
| 80 pontos | Articula as partes do texto, de forma insuficiente, com muitas inadequações, e apresenta repertório limitado de recursos coesivos. |
| 40 pontos | Articula as partes do texto de forma precária. |
| 0 ponto | Não articula as informações. |

In [ ]:
class C4ResponseFormat(CompetenceResponseFormat):
	competence_id: Literal[4]

C4_SYS_PROMPT = "Avalie este trecho de acordo com a competência 4 do ENEM. O tema é 'Desafios para valorização do SUS'. Dê resposta curtíssima, máximo 10 palavras."

C4_TOOLS = [extract_connectives]

C4_AGENT = create_agent(
	model=specialist_model,
	tools=C4_TOOLS,
	system_prompt=C4_SYS_PROMPT,
	response_format=CompetenceResponseFormat
)

In [ ]:
res = C4_AGENT.invoke({
	"messages": [("user", "Drauzio Varella afirma: 'ser ou não ser, eis a questão' para justificar o sucateamento do SUS")]
})
res.keys()

In [ ]:
res['messages']

In [ ]:
C4_OUTPUT = C4ResponseFormat(competence_id=4, **res['structured_response'].model_dump())
print(C4_OUTPUT)

#### 5th Competence Team

Competência V - Proposta de Intervenção

Avalia a elaboração de uma proposta de intervenção para o problema abordado no texto, respeitando os direitos humanos. <br/>
Analisa se a proposta está relacionada ao tema, se é detalhada e se está articulada à discussão desenvolvida ao longo da redação.

Aspectos avaliados
- **presença de proposta de intervenção:** indicação clara de uma ação voltada a enfrentar o problema discutido no texto. Apenas constatar a falta de uma ação ou de um projeto, ou apresentar soluções vagas, genéricas ou inviáveis, não configura proposta de intervenção
- **elementos da proposta:** a proposta é construída a partir da resposta a perguntas como:
  1. **ação:** o que é possível propor como solução para o problema?
  2. **agente:** quem deve executar a ação (governo, ministérios, escolas, mídia, sociedade civil, etc.)?
  3. **meio/modo de execução:** como a ação será posta em prática?
  4. **finalidade/efeito:** qual é o efeito esperado da ação?
  5. **detalhamento:** que outras informações são relevantes para a proposta (exemplificação, explicação, etc.)?
- **relação com o tema:** a proposta responde ao problema específico apresentado no tema, e não apenas ao assunto geral
- **articulação com a discussão:** a proposta é coerente com o ponto de vista e com os argumentos desenvolvidos, respondendo aos problemas abordados ao longo do texto
- **respeito aos direitos humanos:** a proposta não pode violar direitos humanos, como a defesa de tortura, mutilação, execução sumária, "justiça com as próprias mãos" ou qualquer forma de violência ou discriminação por raça, etnia, gênero, credo, opinião política, condição física, origem geográfica ou socioeconômica

> #### Atenção!
> Propostas que desrespeitem os direitos humanos receberão nota 0 (zero) na Competência V.

---

| Pontuação | Justificativa |
|-----------|---|
| 200 pontos | Elabora muito bem proposta de intervenção, detalhada, relacionada ao tema e articulada à discussão desenvolvida no texto. |
| 160 pontos | Elabora bem proposta de intervenção relacionada ao tema e articulada à discussão desenvolvida no texto. |
| 120 pontos | Elabora, de forma mediana, proposta de intervenção relacionada ao tema e articulada à discussão desenvolvida no texto. |
| 80 pontos | Elabora, de forma insuficiente, proposta de intervenção relacionada ao tema, ou não articulada com a discussão desenvolvida no texto. |
| 40 pontos | Apresenta proposta de intervenção vaga, precária ou relacionada apenas ao assunto. |
| 0 ponto | Não apresenta proposta de intervenção ou apresenta proposta não relacionada ao tema ou ao assunto. |

In [ ]:
class C5ResponseFormat(CompetenceResponseFormat):
	competence_id: Literal[5]

C5_SYS_PROMPT = "Avalie este trecho de acordo com a competência 5 do ENEM. O tema é 'Desafios para valorização do SUS'. Dê resposta curtíssima, máximo 10 palavras."

C5_TOOLS = []

C5_AGENT = create_agent(
	model=specialist_model,
	# tools=C5_TOOLS,
	system_prompt=C5_SYS_PROMPT,
	response_format=CompetenceResponseFormat
)

In [ ]:
res = C5_AGENT.invoke({
	"messages": [("user", "Drauzio Varella afirma: 'ser ou não ser, eis a questão' para justificar o sucateamento do SUS")]
})
res.keys()

In [ ]:
res['messages']

In [ ]:
C5_OUTPUT = C5ResponseFormat(competence_id=5, **res['structured_response'].model_dump())
print(C5_OUTPUT)